<a href="https://colab.research.google.com/github/takatakamanbou/AdvML/blob/2025/AdvML2025_ex10notebookA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AdvML ex10notebookA

<img width=72 src="https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/AdvML-logo.png"> [この授業のウェブページ](https://www-tlab.math.ryukoku.ac.jp/wiki/?AdvML)




板書や口頭で補足する前提なので，この notebook だけでは説明が不完全です．


In [ ]:
# 準備あれこれ
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn
seaborn.set_theme()

from scipy.stats import multivariate_normal
from sklearn.mixture import GaussianMixture

---
## EMアルゴリズムによる GMM のパラメータ推定
---

**混合正規分布モデル** (Gaussian Mixture Model, GMM):

$$
\begin{aligned}
p(\pmb{x}) = \sum_{k=1}^{K} w_k \mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k) \qquad \left( 0 \leq w_k \leq 1 \mbox{かつ} \sum_{k=1}^{K}w_k = 1 \right)
\end{aligned}
$$

$w_k, \pmb{\mu}_k, \Sigma_k$ ($k = 1, 2, \ldots, K$) がモデルパラメータ．


データ $\{ \pmb{x}_n \in \mathbb{R}^{D} \}_{n=1}^{N}$ に GMM を当てはめる問題，すなわち，これらのデータに当てはまる GMM のパラメータを推定する問題を考える．

以下，GMM のパラメータをまとめて $\pmb{\theta}$ という記号で表すことにすると，対数尤度 $L(\pmb{\theta})$ は次のようになる．

$$
\begin{aligned}
L(\pmb{\theta}) &= \log \left( \prod_{n=1}^N p_{\pmb{\theta}}(\pmb{x}_n) \right) \\
&= \sum_{n=1}^N \log{ p_{\pmb{\theta}}(\pmb{x}_n) } \\
&= \sum_{n} \log \left( \sum_{z_n} p_{\pmb{\theta}}(\pmb{x}_n, z_n)  \right)
\end{aligned}
$$

ここで，$p_{\pmb{\theta}}(\pmb{x})$ は，上記の $p(\pmb{x})$ と同じものを指す．$\pmb{\theta}$ がパラメータであることを分かりやすくするために添字として $\pmb{\theta}$ を付けている．


単一の正規分布の場合のようにこの対数尤度の最大化（最尤推定）を考えたいが，実はこの式のように $\log$ の中に $\sum$ が入った形をしているものは解析的に解くのが難しい．

---
### 対数尤度 = ELBO + KL-divergence

対数尤度 $L(\pmb{\theta})$ の最大化を実現する方法を探るために，式を変形してみる．
ここでは，簡単のため，ひとつのデータの対数尤度 $\log p_{\pmb{\theta}}(\pmb{x})$ について考える（添字 $n$ も省略）．

任意の確率分布 $q(z)$ に対して，$\log p_{\pmb{\theta}}(\pmb{x})$ は次のように式変形できる．

$$
\begin{aligned}
\log{ p_{\pmb{\theta}}(\pmb{x}) } &= \log{ p_{\pmb{\theta}}(\pmb{x}) } \sum_z q(z) \qquad  \because \sum_z q(z) = 1 \\
&= \sum_z q(z) \log p_{\pmb{\theta}}(\pmb{x}) \\
&= \sum_z q(z) \log \frac{p_{\pmb{\theta}}(\pmb{x}, z)}{p_{\pmb{\theta}}(z|\pmb{x})} \qquad  \because p_{\pmb{\theta}}(\pmb{x}, z) = p_{\pmb{\theta}}(z|\pmb{x})p_{\pmb{\theta}}(\pmb{x}) \\
&= \sum_z q(z) \log \left( \frac{p_{\pmb{\theta}}(\pmb{x}, z)}{p_{\pmb{\theta}}(z|\pmb{x})} \cdot \frac{q(z)}{q(z)} \right) \\
&= \underbrace{\sum_z q(z) \log \frac{p_{\pmb{\theta}}(\pmb{x}, z)}{q(z)}}_{{\rm ELBO}(\pmb{x}; \pmb{\theta}, q)} + \underbrace{\sum_z q(z) \log \frac{q(z)}{p_{\pmb{\theta}}(z|\pmb{x})}}_{D_{\rm KL}(q(z) \Vert p_{\pmb{\theta}}(z|\pmb{x}))}\\
&= {\rm ELBO}(\pmb{x}; \pmb{\theta}, q) + D_{\rm KL}(q(z) \Vert p_{\pmb{\theta}}(z|\pmb{x}))
\end{aligned}
$$

ここで，最後の式の第2項は，「$q(z)$ の $p_{\pmb{\theta}}(z|\pmb{x})$ に対するKLダイバージェンス」と呼ばれる量で，$0$ 以上の実数値をとる．
そのため，
$\log p_{\pmb{\theta}}(\pmb{x}) \geq {\rm ELBO}(\pmb{x}; \pmb{\theta}, q)$
が成り立つ．
すなわち，${\rm ELBO}(\pmb{x}; \pmb{\theta}, q)$ は $\log p_{\pmb{\theta}}(\pmb{x})$ の下界である（注）．
したがって，$q(z)$を適当に定めたうえで ${\rm ELBO}(\pmb{x}; \pmb{\theta}, q)$ を大きくすれば，$\log p_{\pmb{\theta}}(\pmb{x})$ も大きくなる．
また，その式は $\log p_{\pmb{\theta}}(\pmb{x})$ よりも扱いやすい形をしている．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: ELBO = Evidence Lower BOund, エビデンスの下界
</span>







---
### KL-divergence

**KLダイバージェンス** （KL情報量， Kullback-Leibler divergence）は，2つの確率分布の「遠さ」を表す指標．


連続確率分布 $p(x), q(x)$ において，$p(x)$ の $q(x)$ に対する KL ダイバージェンスは

$$
D_{\rm KL}(p \Vert q) = \int_{-\infty}^{\infty} p(x) \log \frac{p(x)}{q(x)} dx
$$

と定義される．また，離散確率分布 $p(x), q(x)$ において，$p(x)$ の $q(x)$ に対する KL ダイバージェンスは

$$
D_{\rm KL}(p \Vert q) = \sum_{x} p(x) \log \frac{p(x)}{q(x)}
$$

と定義される．

次の性質がある．

- $D_{\rm KL}(p \Vert q) \geq 0$．等号が成り立つのは2つの分布が一致する場合かつその場合に限られる
- 一般に $D_{\rm KL}(p \Vert q) \ne D_{\rm KL}(q \Vert p)$ となる．

直感的には，2つの確率分布の遠さを表す距離のような量と見なすことができるが，上記の2つ目のことから分かるように，距離としての性質は満たさない（対称性を満たさない）ことに注意が必要である．

---
### EM アルゴリズム

上述の式変形の結果を利用して対数尤度 $\log p_{\pmb{\theta}}(\pmb{x})$ を最大化するためには，元々のパラメータ $\pmb{\theta}$ に加えて $q(z)$ もパラメータとして最適化する必要がある．
しかし，この最適化は，「$\pmb{\theta}$を固定して $q(z)$ を更新する」手続き（「E-step」と呼ばれる）と，「$q(z)$を固定して$\pmb{\theta}$を更新する」手続き（M-step）を交互に繰り返すことで実現できることが知られている．

- E-step では，$\pmb{\theta}$ が固定されているので，$q(z)$ をどのように選んでも $\log p_{\pmb{\theta}}(\pmb{x})$ は変化しない．このとき， 新しい $q(z)$ を $q(z) = p_{\pmb{\theta}}(z|\pmb{x})$ ととれば， $D_{\rm KL}(q(z) \Vert p_{\pmb{\theta}}(z|\pmb{x})) = 0 $ となって ${\rm ELBO}(\pmb{x}; \pmb{\theta}, q)$ が最大となる．
- M-step では，$q(z)$ を上記で求めた値に固定して，${\rm ELBO}(\pmb{x}; \pmb{\theta}, q)$ を最大にする $\pmb{\theta}$ を求める．


E-step と M-step を交互に繰り返して対数尤度を最大化するこの最適化のアルゴリズムを，**EMアルゴリズム** (Expectation-Maximization Algorithm) という．


ここまでの説明では，一つのデータに対する対数尤度 $\log p_{\pmb{\theta}}(\pmb{x})$ の最大化を考えていた．しかし，実際には，複数のデータに対する対数尤度
$L(\pmb{\theta}) =\sum_{n=1}^N \log p_{\pmb{\theta}}(\pmb{x}_n)$
を最大化したい．
この場合，EMアルゴリズムの手続きは次のようになる（導出は省略）．

1. モデルパラメータ $\pmb{\theta}$ を初期化する．
1. E-step: 現在の $\pmb{\theta}$ を用いて $q_1, q_2, \ldots, q_N$ を更新する．
1. M-step: 次の式を最大にする $\pmb{\theta}$ を求める．
$$
\sum_{n=1}^N {\rm ELBO}(\pmb{x}_n; \pmb{\theta}, q_n)
$$
1.　 繰り返しの終了条件を満たしていなければ 2. へ戻る


終了条件は，「$L(\pmb{\theta})$ の変化量が既定値以下になった」や「繰り返しが既定の回数に達した」等とするのが一般的．

EMアルゴリズムでは，パラメータの更新を繰り返すごとに対数尤度 $L(\pmb{\theta})$ が単調に増加する．すなわち，
$t$ 回目の E-step, M-step によって得られたパラメータを $\pmb{\theta}_{t}$ とおき，この値を用いて次の E-step, M-step を実行して得られるパラメータを $\pmb{\theta}_{t+1}$ とおくと，
$L(\pmb{\theta}_{t+1}) \geq L(\pmb{\theta}_{t})$ が成り立つ．



EMアルゴリズムの概要を説明してきたが，実は，ここまで具体的な GMM の式は出てきていない．
ここまでの議論は，GMM に限らず，潜在変数を持つモデル一般に成り立つものである．


モデルを GMM に限定してより具体的なアルゴリズムを導出する過程の説明は省略する．
結論としては，次のような手続きとなる．

E-step: 現在のパラメータ $\pmb{\theta}$ つまり $w_k, \pmb{\mu}_k, \Sigma_k$ を用いて $q_n(z = k) = p_{\pmb{\theta}}(z = k|\pmb{x})$ を求める．
以下，簡単のため，$q_n(z = k)$ を $q_{n,k}$ と表記する．
$q_{n, k}$ は，$\pmb{x}_n$ が $k$ 番目の正規分布に所属する確率（その正規分布から生成された確率）の推定値とみなせる．

$$
q_{n, k} = \frac{w_k \mathrm{N}(\pmb{x}_n; \pmb{\mu}_k, \Sigma_k)}{\sum_{j=1}^K w_j \mathrm{N}(\pmb{x}_n; \pmb{\mu}_j, \Sigma_j)}
$$


M-step: 求めた $q_{n, k}$ を用いて，$w_k, \pmb{\mu}_k, \Sigma_k$ を更新する．

$$
\begin{aligned}
w_k^{\rm new} &= \frac{1}{N}\sum_{n=1}^N q_{n, k}\\
\pmb{\mu}_k^{\rm new} &= \frac{\sum_{n=1}^{N} q_{n, k} \pmb{x}_n}{\sum_{n=1}^{N} q_{n, k}} \\
\Sigma_k^{\rm new} &= \frac{\sum_{n=1}^{N} q_{n, k} (\pmb{x}_n - \pmb{\mu}_k)(\pmb{x}_n - \pmb{\mu}_k)^{\top}}{\sum_{n=1}^{N} q_{n, k}}
\end{aligned}
$$



### 実験: 2次元データへのGMMの当てはめ

In [ ]:
# データの入手
df = pd.read_csv('https://www-tlab.math.ryukoku.ac.jp/~takataka/course/AdvML/2dim3class.csv')
X = df.drop(columns='label').to_numpy()

In [ ]:
# グラフを描く
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=5)
xmin, xmax = -5, 5
ymin, ymax = -5, 5
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')
plt.show()

In [ ]:
K = 3
gmm = GaussianMixture(n_components=K, covariance_type='full')
gmm.fit(X)

for k in range(K):
    print(f'### {k}-th component')
    print('weight = ', gmm.weights_[k])
    print('mu = ', gmm.means_[k])
    print('cov = ')
    print(gmm.covariances_[k])
    print()

In [ ]:
# グラフ描画用のグリッドデータの作成
x_mesh, y_mesh = np.mgrid[xmin:xmax:(xmax-xmin)/100, ymin:ymax:(ymax-ymin)/100]
X_mesh = np.dstack((x_mesh, y_mesh))

# グラフ
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=8)

# Gaussian を当てはめた結果
for k in range(K):
    mu = gmm.means_[k]
    cov = gmm.covariances_[k]
    ax.contour(x_mesh, y_mesh, multivariate_normal.pdf(X_mesh, mean=mu, cov=cov))
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')

[sklearn.mixture.GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) を読んで，`K` や `covariance_type` を変えて実験してみよう．

---
## 教師なし学習（復習）
---


**教師あり学習** (supervised learning): 個々の学習データが，「入力」とそれに対する「出力の正解」のペアとして与えられる．

**教師なし学習** (unsupervised learning): 学習データは入力のみで構成され，出力の正解は与えられない．

教師なし学習の目的は，大量のデータが与えられたときに，そのデータのもつ規則性や構造を見つけ出すこと．それらを知り，データから有用な情報を抽出することを目指す．
教師あり学習では出力の正解が学習データとして与えられたが，
教師なし学習では，学習データは入力となるもののみが与えられ，出力の正解は与えられない．
データからどのような規則性や構造を見出したいかによって様々な問題設定や手法があり，出力がどのようなものになるかはその選択による．


教師なし学習の問題の代表例

- **クラスタリング**(clustering)
- **次元削減**(次元圧縮とも，dimensionality reduction)
- **確率密度推定**(probability density estimation)


---
## クラスタリング

---


**クラスタリング** (clustering) は，大量のデータをいくつかの塊（**クラスタ** (cluster)）に分ける手続き．
クラスタリングの手法は，**階層型クラスタリング** と **非階層型クラスタリング** に大別できる．

**階層型クラスタリング** (hierarchical clustering): 「クラスタAとクラスタBをあわせたものがクラスタPで，クラスタPとクラスタQをあわせたものがクラスタR」というように，階層的になったクラスタを作るクラスタリング手法．

**非階層型クラスタリング** (non-hierarchical clustering): クラスタ同士に上記のような階層構造をつくらないクラスタリング手法．今回と次回にいくつかの手法を紹介する．

それぞれ，実際の手法には様々なものがあり，データや問題の性質に応じて使い分けられる．
この授業では，階層型クラスタリングについては説明しない（学部2年次科目「[多変量解析及び演習](https://www-tlab.math.ryukoku.ac.jp/wiki/?MVA)」参照．[MVA2024では第13回](https://www-tlab.math.ryukoku.ac.jp/wiki/?MVA/2024#ex13)）．



クラスタリング手法は，データをどのようにクラスタへ振り分けるかという視点で分類することもできる．

- hard-assignment: 個々のデータ点をどれか一つのクラスタにのみ振り分ける．クラスタリングの代表的手法である K-means アルゴリズム（次回解説予定）は，非階層型で hard-assignment な手法．
- soft-assignment: 個々のデータ点を確率的にクラスタに振り分ける．以下で解説する GMM を用いたクラスタリング手法は，非階層型で soft-assignment な手法の代表例．



---
### GMM を用いたクラスタリング

データに $K$ 個の正規分布から成る GMM

$$
\begin{aligned}
p(\pmb{x}) = \sum_{k=1}^{K} w_k \mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k)
\end{aligned}
$$

を当てはめることで，データのクラスタリングができる．データに GMM を当てはめたのち，個々のデータ点がどの正規分布から生成されたかを表す確率を求め，その値によって各データ点をクラスタに振り分ける．クラスタの数 $K$ をいくつにするかは自動的には決められないので，あらかじめ何らかの方法で決める必要がある（注）．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: データから $K$ を推定する手法や，$K$を可変とした GMM の改良手法などもあるが，この授業では説明しない．</br>




GMM において，データが $K$ 個の正規分布のうちどれから生成されたかを表す潜在変数を $z$ とおくと，

$$
\begin{aligned}
p(z = k) &= w_k\\
p(\pmb{x}|z = k) &= \mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k)
\end{aligned}
$$

である．よって，

$$
p(\pmb{x}, z = k) = p(\pmb{x}|z = k) p(z = k) = p(z = k|\pmb{x}) p(\pmb{x})
$$

より，

$$
\begin{aligned}
p(z = k|\pmb{x}) &= \frac{p(\pmb{x}|z = k) p(z = k)}{p(\pmb{x})}\\
&= \frac{w_k \mathrm{N}(\pmb{x}; \pmb{\mu}_k, \Sigma_k)}{\sum_{j=1}^{K} w_j \mathrm{N}(\pmb{x}; \pmb{\mu}_j, \Sigma_j)}
\end{aligned}
$$

が成り立つ．この $p(z = k|\pmb{x}) $ ($k = 1, 2, \ldots, K$) が，データ点 $\pmb{x}$ が $k$ 番目の正規分布が表すクラスタに所属する確率を表す（注）．

<br>
<hr width="50%" align="left">
<span style="font-size: 75%">
※注: $\underset{k}{\operatorname{argmax}}p(z = k|\pmb{x})$ として hard-assignment 化することも可能．

</span>



---
### 実験: GMM を用いた2次元データのクラスタリング

In [ ]:
# 実験用データの入手
df = pd.read_csv('https://www-tlab.math.ryukoku.ac.jp/~takataka/course/ML/data4kmeans.csv')
dat1 = df[['x1', 'x2']].to_numpy()
dat2 = df[['y1', 'y2']].to_numpy()

次の2つのコードセルを実行すると，2次元のデータを GMM を用いてクラスタリングすることができる．

In [ ]:
X = dat1  #　データその1
#X = dat2  # データその2

# GMM の当てはめ
K = 3
gmm = GaussianMixture(n_components=3, covariance_type='full')
gmm.fit(X)

# グラフ描画用のグリッドデータの作成
xmin, xmax = -5, 5
ymin, ymax = -5, 5
x_mesh, y_mesh = np.mgrid[xmin:xmax:(xmax-xmin)/100, ymin:ymax:(ymax-ymin)/100]
X_mesh = np.dstack((x_mesh, y_mesh))

# グラフ
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], s=8)

# Gaussian を当てはめた結果
for k in range(K):
    mu = gmm.means_[k]
    cov = gmm.covariances_[k]
    ax.contour(x_mesh, y_mesh, multivariate_normal.pdf(X_mesh, mean=mu, cov=cov))
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect('equal')

In [ ]:
# 平面上の各点の p(z=k|x) の値を求める
p = gmm.predict_proba(X_mesh.reshape((-1, 2))).T
pp = p.reshape((K, X_mesh.shape[0], X_mesh.shape[1]))

# 学習データをクラスタへ hard-assign
y = gmm.predict(X)
mu = gmm.means_

# グラフ
fig = plt.figure(figsize=(9, 6))
colors = seaborn.color_palette(n_colors=K)

# 平面を p(z=k|x) の値で塗り分け
ax0 = fig.add_subplot(121)
cmap = ['Blues', 'Oranges', 'Greens']
for k in range(K):
    ax0.scatter(X[y==k, 0], X[y==k, 1], s=8)
    ax0.contourf(x_mesh, y_mesh, pp[k], levels=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0], cmap=cmap[k], alpha=0.3)
    ax0.plot(mu[k, 0], mu[k, 1], color='white', marker='*', markerfacecolor=colors[k], markersize=25)
ax0.set_xlim(xmin, xmax)
ax0.set_ylim(ymin, ymax)
ax0.set_aspect('equal')

# 学習データ点の塗り分け
ax1 = fig.add_subplot(122)
colors = seaborn.color_palette(n_colors=K)
for k in range(K):
    ax1.scatter(X[y == k, 0], X[y == k, 1], label=f'cluster {k}', s=8)
    ax1.plot(mu[k, 0], mu[k, 1], color='white', marker='*', markerfacecolor=colors[k], markersize=25)
ax1.set_xlim(xmin, xmax)
ax1.set_ylim(ymin, ymax)
ax1.legend()
ax1.set_aspect('equal')


plt.show()

左図は，平面全体を $p(z=k|\pmb{x})$ の値によって塗り分けたものである．右図は，学習データ点を hard-assigmnent した結果を示している．

以下の行のコメントの付け方を変えて，「データその2」でも実験してみよう．
```
X = dat1  #　データその1
#X = dat2  # データその2
```